In [1]:
# Step 1: Import required libraries
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Step 2: Define file paths for model persistence and inference
MODEL_FILE = "housing_full_pipeline.pkl"
INPUT_FILE = "input.csv"
OUTPUT_FILE = "output.csv"

In [3]:
# Step 3: Load dataset and perform stratified train/test split
housing = pd.read_excel("housing.csv.xlsx")

housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].drop("income_cat", axis=1)
    strat_test_set = housing.loc[test_index].drop("income_cat", axis=1)

# Save the raw unseen test set to input.csv for simulating real-world inference
strat_test_set.to_csv(INPUT_FILE, index=False)
print(f"Training set shape: {strat_train_set.shape}")
print(f"Test set saved to '{INPUT_FILE}' with shape: {strat_test_set.shape}")

Training set shape: (16512, 10)
Test set saved to 'input.csv' with shape: (4128, 10)


In [4]:
# Step 4: Separate predictors and labels for the training set
housing_features = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

# Identify numerical and categorical column names
num_attribs = housing_features.drop("ocean_proximity", axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]
print("Numerical features:", num_attribs)
print("Categorical features:", cat_attribs)

Numerical features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical features: ['ocean_proximity']


In [5]:
# Step 5: Build a single, unified End-to-End Pipeline (Preprocessing + Model)
# 1. Numerical preprocessing pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 2. Categorical preprocessing pipeline
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 3. ColumnTransformer to merge numerical and categorical pipelines
preprocessing_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

# 4. Master Pipeline: Combines Preprocessing AND RandomForestRegressor
# Because the final step is a predictor, this master pipeline implements fit() and predict() directly!
full_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", RandomForestRegressor(random_state=42)),
])

In [6]:
# Step 6: Train the entire pipeline on raw training data and save to a single file
print("Training the end-to-end pipeline on raw features...")
full_pipeline.fit(housing_features, housing_labels)

# Persist the master pipeline (which includes imputer, scaler, encoder, AND model)
joblib.dump(full_pipeline, MODEL_FILE)
print(f"Master pipeline successfully trained and saved to '{MODEL_FILE}'.")

Training the end-to-end pipeline on raw features...
Master pipeline successfully trained and saved to 'housing_full_pipeline.pkl'.


In [7]:
# Step 7: INFERENCE PHASE — Load the saved master pipeline from disk
# Simulating loading the model in an API, web application, or batch prediction script
loaded_pipeline = joblib.load(MODEL_FILE)
print(f"Model pipeline '{MODEL_FILE}' loaded successfully.")

Model pipeline 'housing_full_pipeline.pkl' loaded successfully.


In [8]:
# Step 8: Run inference on unseen input data
input_data = pd.read_csv(INPUT_FILE)

# Separate the actual ground truth and raw predictors
X_test = input_data.drop("median_house_value", axis=1)
y_actual = input_data["median_house_value"].copy()

# The loaded master pipeline handles preprocessing AND prediction automatically in one call!
predictions = loaded_pipeline.predict(X_test)

# Store predictions in a separate column so we can compare actual vs predicted prices
input_data["predicted_house_value"] = predictions
input_data["prediction_error"] = input_data["predicted_house_value"] - input_data["median_house_value"]

# Preview comparison of actual vs predicted values
comparison_cols = ["median_income", "ocean_proximity", "median_house_value", "predicted_house_value", "prediction_error"]
print("Sample Predictions Comparison:")
input_data[comparison_cols].head(10)

Sample Predictions Comparison:


,median_income,ocean_proximity,median_house_value,predicted_house_value,prediction_error
0,8.2816,<1H OCEAN,500001,483020.60,-16980.40
1,5.0099,<1H OCEAN,162500,221708.09,59208.09
2,4.3839,<1H OCEAN,204600,205706.00,1106.00
3,3.2708,<1H OCEAN,159700,170623.00,10923.00
4,4.3529,NEAR OCEAN,184000,212497.00,28497.00
5,4.1667,<1H OCEAN,151900,164798.00,12898.00
6,1.3750,NEAR BAY,104900,114046.00,9146.00
7,7.0001,NEAR OCEAN,500001,434144.28,-65856.72
8,4.4699,<1H OCEAN,367400,280961.03,-86438.97
9,3.9875,NEAR OCEAN,346500,258439.07,-88060.93


In [9]:
# Step 9: Final Model Evaluation on Unseen Test Data
# Evaluate how well the model generalizes to the test set using Root Mean Squared Error (RMSE)
final_rmse = root_mean_squared_error(y_actual, predictions)
print(f"Final Test RMSE: ${final_rmse:,.2f}")

# Display prediction error summary statistics
print("\nPrediction Error Summary ($):")
print(input_data["prediction_error"].describe().round(2))

Final Test RMSE: $47,197.67

Prediction Error Summary ($):
count      4128.00
mean       1291.27
std       47185.72
min     -307948.97
25%      -14501.00
50%        4805.50
75%       22476.75
max      257667.06
Name: prediction_error, dtype: float64


In [10]:
# Step 10: Save inference results with actuals, predictions, and errors to output.csv
input_data.to_csv(OUTPUT_FILE, index=False)
print(f"Inference results with comparison columns successfully saved to '{OUTPUT_FILE}'.")

Inference results with comparison columns successfully saved to 'output.csv'.
